
# Voyage Analytics – Premium DS/ML Internship Project

## Integrating MLOps in Travel Analytics

### Objectives
- Analyze travel customer behavior
- Discover business insights from hotel and flight bookings
- Build customer segmentation using K-Means
- Create a hotel recommendation system
- Prepare the project for MLOps deployment


In [ ]:

# Core Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')


In [ ]:

# Load Dataset

import zipfile

zip_path = 'travel_capstone dataset-20260608T152210Z-3-001.zip'

zf = zipfile.ZipFile(zip_path)

users = pd.read_csv(zf.open('travel_capstone dataset/users.csv'))
hotels = pd.read_csv(zf.open('travel_capstone dataset/hotels.csv'))
flights = pd.read_csv(zf.open('travel_capstone dataset/flights.csv'))

print("Users:", users.shape)
print("Hotels:", hotels.shape)
print("Flights:", flights.shape)


In [ ]:

# Dataset Overview

display(users.head())
display(hotels.head())
display(flights.head())


In [ ]:

# Missing Values

print("Users")
print(users.isnull().sum())

print("\nHotels")
print(hotels.isnull().sum())

print("\nFlights")
print(flights.isnull().sum())


In [ ]:

# Data Types

print(users.dtypes)
print(hotels.dtypes)
print(flights.dtypes)


## Exploratory Data Analysis

In [ ]:

# Numerical Summary

display(users.describe())
display(flights.describe())


In [ ]:

# Age Distribution

if 'age' in users.columns:
    users['age'].hist(bins=20)
    plt.title('Age Distribution')
    plt.show()


In [ ]:

# Gender Distribution

if 'gender' in users.columns:
    users['gender'].value_counts().plot(kind='bar')
    plt.title('Gender Distribution')
    plt.show()


In [ ]:

# Flight Price Analysis

if 'price' in flights.columns:
    flights['price'].hist(bins=30)
    plt.title('Flight Price Distribution')
    plt.show()


## Feature Engineering

In [ ]:

# Customer Travel Statistics

customer_features = flights.groupby('userCode').agg({
    'price':'sum',
    'distance':'sum'
}).reset_index()

customer_features.columns = [
    'userCode',
    'total_spent',
    'total_distance'
]

customer_features.head()


In [ ]:

# Merge User Data

if 'code' in users.columns:
    customer_features = customer_features.merge(
        users,
        left_on='userCode',
        right_on='code',
        how='left'
    )

customer_features.head()


## Customer Segmentation Using K-Means

In [ ]:

features = customer_features.select_dtypes(include=np.number)

scaler = StandardScaler()

scaled = scaler.fit_transform(features.fillna(0))

scores = []

for k in range(2,8):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(scaled)
    scores.append(model.inertia_)

plt.plot(range(2,8), scores, marker='o')
plt.title('Elbow Method')
plt.xlabel('Clusters')
plt.ylabel('Inertia')
plt.show()


In [ ]:

kmeans = KMeans(
    n_clusters=4,
    random_state=42,
    n_init=10
)

customer_features['cluster'] = kmeans.fit_predict(scaled)

customer_features.head()


In [ ]:

customer_features['cluster'].value_counts().plot(kind='bar')
plt.title('Customer Segments')
plt.show()


In [ ]:

cluster_summary = customer_features.groupby('cluster').mean(numeric_only=True)
cluster_summary


## Hotel Recommendation System

In [ ]:

# User-Hotel Matrix

hotel_matrix = pd.crosstab(
    hotels['userCode'],
    hotels['name']
)

hotel_matrix.head()


In [ ]:

# Similarity Matrix

similarity = cosine_similarity(hotel_matrix)

similarity_df = pd.DataFrame(
    similarity,
    index=hotel_matrix.index,
    columns=hotel_matrix.index
)

similarity_df.head()


In [ ]:

def recommend_hotels(user_id, top_n=5):

    similar_users = similarity_df[user_id]         .sort_values(ascending=False)[1:10].index

    visited = set(
        hotels[hotels['userCode']==user_id]['name']
    )

    recommendations = (
        hotels[hotels['userCode'].isin(similar_users)]
        ['name']
        .value_counts()
    )

    recommendations = [
        h for h in recommendations.index
        if h not in visited
    ]

    return recommendations[:top_n]


In [ ]:

# Example Recommendation

sample_user = hotel_matrix.index[0]

recommend_hotels(sample_user)


## Business Insights

In [ ]:

top_customers = customer_features.sort_values(
    'total_spent',
    ascending=False
).head(10)

top_customers[['userCode','total_spent']]


In [ ]:

top_hotels = hotels['name'].value_counts().head(10)

top_hotels.plot(kind='bar')
plt.title('Most Popular Hotels')
plt.show()



# MLOps Production Architecture

### Data Layer
- Flight Data
- Hotel Data
- User Data

### ML Layer
- Feature Engineering
- Customer Segmentation
- Recommendation Engine

### Deployment
- Streamlit Dashboard
- REST API

### Monitoring
- Model Drift Detection
- Automated Retraining
- Performance Monitoring

### CI/CD
- GitHub Actions
- Docker
- Cloud Deployment



# Conclusion

### Achievements
- Performed travel analytics
- Created customer segments
- Built recommendation engine
- Generated business insights
- Designed MLOps-ready architecture

### Future Work
- Deep Learning Recommendations
- Real-Time Personalization
- Dynamic Pricing Models
- Travel Demand Forecasting
